# Runner econométrico — Credit Spreads S&P 500

## Objetivo

Este notebook implementa la estrategia econométrica final de la tesis, cuyo objetivo es analizar cómo los credit spreads de bonos corporativos del S&P 500 reflejan:

1. condiciones macro-financieras comunes,
2. shocks agregados de mercado y crédito,
3. heterogeneidad en la exposición firm-level a dichos shocks.

---

## Estrategia empírica

La estimación sigue una arquitectura híbrida:

- **Backbone + módulos (Arquitectura B):**
  se modelan explícitamente los factores agregados observables.

- **Secuencia M0–M6 (Arquitectura A):**
  los modelos se presentan progresivamente para aislar canales económicos.

- **Robustez en variable dependiente (Arquitectura C):**
  se evalúa la estabilidad de los resultados usando distintas medidas de credit spread.

---

## Variable dependiente

- Principal: `oas_mean`
- Robustez:
  - `spread_mean_bps`
  - `cds_5y_mean` o `cds_5y_eom`

---

## Estructura del notebook

1. Setup y configuración  
2. Carga del panel final  
3. Definición conceptual de variables  
4. Preparación de variables  
5. Construcción de interacciones  
6. Definición de muestras  
7. Funciones de estimación  
8. Modelos M0–M6  
9. Estimación principal (OAS)  
10. Interpretación económica  
11. Export de resultados  
12. Robustez (dependiente)  
13. Robusteces adicionales  
14. QA final  
15. Cierre  

---

## Principios clave

- Efectos fijos por firma en todos los modelos  
- Efectos fijos de tiempo solo en el benchmark (M0)  
- Errores clusterizados por firma  
- `ivol_252` se incluye siempre y no se interactúa  
- Market power se utiliza solo como modulador (interacciones)  
- Separación explícita entre:
  - shocks agregados
  - exposición diferencial

## 1) Setup y configuración

En esta sección se definen:

- librerías necesarias,
- paths del proyecto,
- configuración de warnings y display,
- directorios de output.

El objetivo es asegurar que el notebook sea completamente reproducible y consistente con la estructura del repositorio.

In [17]:
# ==========================================================
# 1. SETUP Y CONFIGURACIÓN
# ==========================================================

# ------------------------------
# Librerías
# ------------------------------
import pandas as pd
import numpy as np

from pathlib import Path
import warnings

# Modelos econométricos
from linearmodels.panel import PanelOLS

# ------------------------------
# Configuración general
# ------------------------------
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# ------------------------------
# Paths del proyecto
# ------------------------------
BASE_DIR = Path.cwd().resolve().parents[0]

DATA_DIR = BASE_DIR / "data"
CLEAN_DIR = DATA_DIR / "clean"

OUTPUT_DIR = BASE_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"
RESULTS_DIR = OUTPUT_DIR / "results"

# Crear carpetas si no existen
TABLES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------
# Parámetros globales
# ------------------------------
CLUSTER_ENTITY = True   # cluster por firma
USE_TIME_FE_M0 = True   # FE tiempo solo en M0

print("Setup completo.")


Setup completo.


## 2) Carga del panel final

En esta sección se carga el panel final construido en el Notebook 2.

El panel contiene información a nivel firma-tiempo e incluye:

- variables dependientes (OAS, spreads, CDS),
- shocks agregados (mercado y crédito),
- variables macroeconómicas,
- controles firm-level,
- proxies de market power.

---

## Validaciones realizadas

Se realizan chequeos básicos para asegurar:

- correcta carga del archivo,
- dimensiones del panel,
- presencia de columnas clave,
- consistencia de tipos de datos,
- unicidad de identificador firma-tiempo.

---

## Output esperado

Un DataFrame limpio, consistente y listo para:

- definición de variables,
- construcción de muestras,
- estimación econométrica.

### Estandarización de nombres de variables

Se renombran columnas para evitar problemas en la estimación:

- se eliminan espacios,
- se reemplazan caracteres especiales,
- se homogeniza formato snake_case.

Esto es necesario para compatibilidad con linearmodels y fórmulas econométricas.

In [32]:
# ==========================================================
# 2. CARGA DEL PANEL FINAL
# ==========================================================

# ------------------------------
# Path del panel
# ------------------------------
PANEL_PATH = CLEAN_DIR / "panel_master.parquet"

# ------------------------------
# Carga
# ------------------------------
print("Cargando panel desde:", PANEL_PATH)
df = pd.read_parquet(PANEL_PATH)
print("Panel cargado correctamente.")

# ------------------------------
# Dimensiones iniciales
# ------------------------------
print("\nDimensión del panel:")
print(df.shape)

# ------------------------------
# Columnas disponibles (raw)
# ------------------------------
print("\nColumnas del panel (raw):")
print(sorted(df.columns.tolist()))

# ------------------------------
# Estandarización de nombres de columnas
# ------------------------------
def clean_column_names(columns):
    return (
        columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^\w]+", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )

df.columns = clean_column_names(df.columns)

print("\nColumnas estandarizadas:")
print(sorted(df.columns.tolist()))

# ------------------------------
# Tipos de datos
# ------------------------------
print("\nTipos de datos:")
print(df.dtypes.head(20))

# ------------------------------
# Conversión de fecha
# ------------------------------
if "date" not in df.columns:
    raise ValueError("No existe la columna 'date' en el panel.")

if not pd.api.types.is_datetime64_any_dtype(df["date"]):
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

# ------------------------------
# Chequeo de identificadores clave
# ------------------------------
required_ids = ["issuer", "date"]
missing_ids = [c for c in required_ids if c not in df.columns]

if missing_ids:
    raise ValueError(f"Faltan columnas clave del panel: {missing_ids}")

# ------------------------------
# Unicidad firma-tiempo
# ------------------------------
duplicates = df.duplicated(subset=["issuer", "date"]).sum()
print(f"\nDuplicados (issuer-date): {duplicates}")

# ------------------------------
# Orden del panel
# ------------------------------
df = df.sort_values(["issuer", "date"]).reset_index(drop=True)

# ------------------------------
# Chequeo de valores faltantes clave
# ------------------------------
key_vars = [
    "oas_mean",
    "spread_mean_bps",
    "cds_5y_mean",
    "mkt_ret",
    "credit_level",
    "credit_slope"
]

missing_key_vars = [c for c in key_vars if c not in df.columns]
if missing_key_vars:
    raise ValueError(f"Faltan variables clave del panel: {missing_key_vars}")

missing_summary = df[key_vars].isna().mean().sort_values(ascending=False)

print("\n% de missing en variables clave:")
print(missing_summary)

# ------------------------------
# Chequeo de market power
# ------------------------------
market_power_vars = [
    "market_share_raw",
    "market_share_w",
    "market_share_sector",
    "market_share_industry_group",
    "market_share_industry",
    "market_share_subindustry",
]

available_mp = [c for c in market_power_vars if c in df.columns]
missing_mp = [c for c in market_power_vars if c not in df.columns]

print("\nProxies de market power disponibles:")
print(available_mp)

if missing_mp:
    print("\nProxies de market power faltantes:")
    print(missing_mp)

if available_mp:
    print("\n% de missing en proxies de market power:")
    print(df[available_mp].isna().mean().sort_values(ascending=False))

# ------------------------------
# Chequeo rápido de panel
# ------------------------------
n_firms = df["issuer"].nunique()
n_periods = df["date"].nunique()

print("\nEstructura del panel:")
print(f"- Firmas: {n_firms}")
print(f"- Periodos: {n_periods}")
print(f"- Observaciones totales: {len(df)}")

print("\nCarga y validación inicial completadas.")

Cargando panel desde: C:\Users\vidar\OneDrive\Desktop\UDESA\Tesis\Tesis-main\data\clean\panel_master.parquet
Panel cargado correctamente.

Dimensión del panel:
(2788, 55)

Columnas del panel (raw):
['US1M', 'US1Y', 'US30Y', 'US3Y', 'US5Y', 'US6M', 'US7Y', 'beta_252', 'cash_to_assets', 'cds_5y_eom', 'cds_5y_mean', 'covid_dummy', 'crc_level_beta', 'crc_nobs', 'crc_r2', 'crc_slope_beta', 'credit_level', 'credit_slope', 'current_ratio', 'date', 'interest_coverage', 'issuer', 'ivol_252', 'ivol_sector', 'leverage', 'log_assets', 'ltdebt_share', 'market_share_industry', 'market_share_industry_group', 'market_share_raw', 'market_share_sector', 'market_share_subindustry', 'market_share_w', 'mkt_ret', 'mkt_vol_60d', 'move_eom', 'n_bonds', 'n_bonds_oas', 'n_obs_cds', 'oas_mean', 'oas_median', 'policy_rate', 'residual_maturity_mean', 'rollover_12m', 'sector', 'sector_ric', 'spread_mean_bps', 'term_spread_10y_2y', 'term_spread_10y_3m', 'ticker', 'ticker_base', 'ust_10y', 'ust_2y', 'ust_3m', 'ytm_me

In [33]:
print("\n" + "="*60)
print("DEBUG MARKET POWER")
print("="*60)

# 1) Ver todas las columnas
print("\nColumnas en df:")
print(df.columns.tolist())

# 2) Buscar columnas candidatas relacionadas a market power
print("\nColumnas que contienen 'market', 'share', 'power':")
candidates = [c for c in df.columns if any(x in c.lower() for x in ["market", "share", "power"])]
print(candidates)

# 3) Chequeo directo de la variable esperada
print("\n¿Existe 'market_share_industry'?")
print("market_share_industry" in df.columns)

# 4) Si existe, ver contenido
if "market_share_industry" in df.columns:
    print("\nHead de market_share_industry:")
    print(df["market_share_industry"].head())

    print("\nDescribe:")
    print(df["market_share_industry"].describe())

    print("\nNaN share:")
    print(df["market_share_industry"].isna().mean())

# 5) Chequear otras posibles variables de market power típicas
possible_vars = [
    "market_share",
    "market_share_sector",
    "market_share_industry_group",
    "hhi",
    "hhi_industry",
    "relative_size",
    "markup",
]

print("\nChequeo de variables alternativas posibles:")
for v in possible_vars:
    print(f"{v}: {v in df.columns}")


DEBUG MARKET POWER

Columnas en df:
['issuer', 'date', 'spread_mean_bps', 'ytm_mean', 'residual_maturity_mean', 'n_bonds', 'ticker', 'sector', 'ticker_base', 'beta_252', 'ivol_252', 'log_assets', 'leverage', 'cash_to_assets', 'current_ratio', 'interest_coverage', 'ltdebt_share', 'rollover_12m', 'market_share_raw', 'market_share_w', 'market_share_sector', 'market_share_industry_group', 'market_share_industry', 'market_share_subindustry', 'crc_level_beta', 'crc_slope_beta', 'crc_r2', 'crc_nobs', 'mkt_ret', 'mkt_vol_60d', 'credit_level', 'credit_slope', 'policy_rate', 'term_spread_10y_2y', 'move_eom', 'ust_3m', 'ust_2y', 'ust_10y', 'term_spread_10y_3m', 'us1m', 'us1y', 'us30y', 'us3y', 'us5y', 'us6m', 'us7y', 'covid_dummy', 'sector_ric', 'ivol_sector', 'oas_mean', 'oas_median', 'n_bonds_oas', 'cds_5y_mean', 'cds_5y_eom', 'n_obs_cds']

Columnas que contienen 'market', 'share', 'power':
['ltdebt_share', 'market_share_raw', 'market_share_w', 'market_share_sector', 'market_share_industry_gro

## 3) Definición conceptual de variables

Esta sección define la estructura económica del modelo empírico.

La especificación distingue explícitamente entre:

1. Variable dependiente
2. Controles firm-level
3. Riesgo idiosincrático (zero-beta)
4. Entorno macro-financiero (backbone)
5. Shocks agregados (mercado y crédito)
6. Heterogeneidad en la exposición (interacciones)

---

## Principio de identificación

La estrategia empírica separa tres componentes fundamentales:

- Factores agregados comunes (macro)
- Shocks agregados observables (mercado y crédito)
- Exposición diferencial entre firmas

Esto permite evitar interpretaciones erróneas en las que variables firm-level capturan shocks agregados.

---

## Variable dependiente

- Principal: `oas_mean`
- Robustez:
  - `spread_mean_bps`
  - `cds_5y_mean` / `cds_5y_eom`

---

## Bloques del modelo

### Controles firm-level
Capturan características estructurales de las firmas.

### Riesgo idiosincrático
`ivol_252` se incluye como componente zero-beta.

### Backbone macro-financiero
Variables agregadas que reemplazan efectos fijos de tiempo.

### Shocks agregados
Capturan innovaciones en el entorno de mercado y crédito.

### Heterogeneidad
Interacciones que permiten exposición diferencial a shocks.

In [34]:
# ==========================================================
# 3.1 VARIABLE DEPENDIENTE
# ==========================================================

DEP_VAR_MAIN = "oas_mean"

DEP_VARS_ALT = [
    "spread_mean_bps",
    "cds_5y_mean",   # alternativa posible más adelante: cds_5y_eom
]

# ==========================================================
# 3.2 CONTROLES FIRM-LEVEL
# ==========================================================

FIRM_CONTROLS = [
    "leverage",
    "log_assets",
    "cash_to_assets",
    "current_ratio",
    "interest_coverage",
    "residual_maturity_mean",
    "rollover_12m"
]

missing_controls = [c for c in FIRM_CONTROLS if c not in df.columns]
if missing_controls:
    raise ValueError(f"Faltan controles firm-level ya construidos en el panel: {missing_controls}")

print("✔ Controles firm-level correctamente detectados")

# ==========================================================
# 3.3 RIESGO IDIOSINCRÁTICO
# ==========================================================

IVOL_VAR = "ivol_252"

if IVOL_VAR not in df.columns:
    raise ValueError(f"Falta variable de riesgo idiosincrático: {IVOL_VAR}")

print("✔ Riesgo idiosincrático correctamente detectado")

# ==========================================================
# 3.4 BACKBONE MACRO-FINANCIERO
# ==========================================================

MACRO_VARS = [
    "policy_rate",
    "term_spread_10y_2y",
    "move_eom"
]

missing_macro = [c for c in MACRO_VARS if c not in df.columns]
if missing_macro:
    raise ValueError(f"Faltan variables macro-financieras: {missing_macro}")

print("✔ Backbone macro-financiero correctamente detectado")

# ==========================================================
# 3.5 SHOCKS AGREGADOS
# ==========================================================

MARKET_SHOCK = "mkt_ret"

CREDIT_VARS = [
    "credit_level",
    "credit_slope"
]

for v in [MARKET_SHOCK] + CREDIT_VARS:
    if v not in df.columns:
        raise ValueError(f"Falta variable clave del modelo: {v}")

print("✔ Shocks agregados correctamente detectados")

# ==========================================================
# 3.6 MARKET POWER
# ==========================================================

# Proxy principal sugerida
MARKET_POWER = "market_share_industry_group"

# Set completo de proxies candidatas
MARKET_POWER_PROXIES = [
    "market_share_sector",
    "market_share_industry_group",
    "market_share_industry",
    "market_share_subindustry",
    "market_share_w",
    "market_share_raw",
]

available_market_power = [v for v in MARKET_POWER_PROXIES if v in df.columns]
missing_market_power = [v for v in MARKET_POWER_PROXIES if v not in df.columns]

print("\nProxies de market power detectadas en el panel:")
print(available_market_power)

if missing_market_power:
    print("\nProxies de market power no disponibles:")
    print(missing_market_power)

if MARKET_POWER not in df.columns:
    raise ValueError(f"Falta proxy principal de market power: {MARKET_POWER}")

print(f"✔ Proxy principal de market power correctamente detectada: {MARKET_POWER}")

# Debug útil: missingness y dispersión
print("\nResumen rápido de proxies de market power disponibles:")
for v in available_market_power:
    print(f"\n--- {v} ---")
    print("NaN share:", round(df[v].isna().mean(), 4))
    print("std:", round(df[v].std(skipna=True), 6))
    print("p95:", round(df[v].quantile(0.95), 6))
    print("max:", round(df[v].max(skipna=True), 6))

# ==========================================================
# 3.7 CHEQUEO FINAL DE VARIABLES DEL MODELO
# ==========================================================

model_core_vars = (
    [DEP_VAR_MAIN, IVOL_VAR]
    + FIRM_CONTROLS
    + MACRO_VARS
    + [MARKET_SHOCK]
    + CREDIT_VARS
    + [MARKET_POWER]
)

missing_model_core = [v for v in model_core_vars if v not in df.columns]

if missing_model_core:
    raise ValueError(f"Faltan variables core del modelo: {missing_model_core}")

print("✔ Todas las variables core del modelo están presentes")

✔ Controles firm-level correctamente detectados
✔ Riesgo idiosincrático correctamente detectado
✔ Backbone macro-financiero correctamente detectado
✔ Shocks agregados correctamente detectados

Proxies de market power detectadas en el panel:
['market_share_sector', 'market_share_industry_group', 'market_share_industry', 'market_share_subindustry', 'market_share_w', 'market_share_raw']
✔ Proxy principal de market power correctamente detectada: market_share_industry_group

Resumen rápido de proxies de market power disponibles:

--- market_share_sector ---
NaN share: 0.1212
std: 0.073447
p95: 0.264541
max: 0.596055

--- market_share_industry_group ---
NaN share: 0.1212
std: 0.143725
p95: 0.509066
max: 0.797306

--- market_share_industry ---
NaN share: 0.1212
std: 0.17807
p95: 0.611636
max: 0.922266

--- market_share_subindustry ---
NaN share: 0.1212
std: 0.183568
p95: 0.619494
max: 0.972295

--- market_share_w ---
NaN share: 0.1212
std: 0.012816
p95: 0.024906
max: 0.26766

--- market_share

## 4) Preparación de variables para estimación

En esta sección se prepara el panel para la estimación econométrica.

Se realizan los siguientes pasos:

1. Tratamiento de valores extremos
2. Manejo de valores faltantes
3. Validación de soporte de variables
4. Preparación del panel para modelos

---

## Principios econométricos

- No se imputan valores faltantes
- Los ratios se definen solo en su soporte válido
- Se controla la influencia de outliers
- Cada modelo utilizará su propia muestra efectiva (drop NA específico)

---

## Resultado esperado

Un panel limpio y consistente que puede ser utilizado para estimaciones con efectos fijos sin introducir sesgos mecánicos.

In [35]:

# ==========================================================
# 4.1 WINSORIZACIÓN
# ==========================================================

def winsorize_series(s, lower=0.01, upper=0.99):
    return s.clip(lower=s.quantile(lower), upper=s.quantile(upper))

# ==========================================================
# 4.1 WINSORIZACIÓN VARIABLES CLAVE
# ==========================================================


# Variables continuas sensibles
WINSOR_VARS = [
    "leverage",
    "cash_to_assets",
    "current_ratio",
    "interest_coverage",
    "rollover_12m",
    "residual_maturity_mean",
    "ivol_252",
    "oas_mean",
    "spread_mean_bps",
    "cds_5y_mean"
]

for var in WINSOR_VARS:
    if var in df.columns:
        df[var] = winsorize_series(df[var])

# ==========================================================
# 4.3 VALIDACIÓN DE SOPORTE
# ==========================================================

# eliminar valores absurdos
df.loc[df["leverage"] < 0, "leverage"] = np.nan

df.loc[df["current_ratio"] <= 0, "current_ratio"] = np.nan

df.loc[df["interest_coverage"] <= 0, "interest_coverage"] = np.nan

# ==========================================================
# 4.4 MISSING VALUES
# ==========================================================

key_vars_check = (
    [DEP_VAR_MAIN]
    + FIRM_CONTROLS
    + MACRO_VARS
    + [IVOL_VAR]
    + [MARKET_SHOCK]
    + CREDIT_VARS
)

missing_summary = df[key_vars_check].isna().mean().sort_values(ascending=False)

print("\n% missing variables clave:")
print(missing_summary)

# ==========================================================
# 4.5 INDEX PANEL
# ==========================================================

df = df.set_index(["issuer", "date"])

df = df.sort_index()

print("✔ Panel listo para estimación")

# ==========================================================
# 4.6 VARIACIÓN WITHIN
# ==========================================================

def check_within_variation(df, var):
    return df.groupby(level=0)[var].nunique().mean()

for var in ["leverage", "ivol_252", "mkt_ret"]:
    if var in df.columns:
        print(f"{var}: variación promedio dentro de firma =", check_within_variation(df, var))




% missing variables clave:
current_ratio             0.226327
interest_coverage         0.212339
leverage                  0.203730
rollover_12m              0.203730
cash_to_assets            0.139885
ivol_252                  0.104376
residual_maturity_mean    0.011119
log_assets                0.007174
credit_level              0.003587
credit_slope              0.003587
oas_mean                  0.000000
policy_rate               0.000000
term_spread_10y_2y        0.000000
move_eom                  0.000000
mkt_ret                   0.000000
dtype: float64
✔ Panel listo para estimación
leverage: variación promedio dentro de firma = 6.15625
ivol_252: variación promedio dentro de firma = 76.625
mkt_ret: variación promedio dentro de firma = 87.125


## 5) Construcción de interacciones

En esta sección se construyen variables de interacción para capturar heterogeneidad en la exposición de las firmas a shocks agregados.

---

## Motivación económica

Las firmas no responden homogéneamente a shocks de mercado o crédito. La exposición depende de características estructurales como:

- apalancamiento,
- riesgo de refinanciación,
- poder de mercado.

---

## Estrategia

Se construyen interacciones específicas y disciplinadas:

- Mercado:
  - mkt_ret × leverage
  - mkt_ret × market power

- Crédito:
  - credit_level × rollover
  - credit_level × market power

Estas interacciones permiten identificar mecanismos de transmisión heterogéneos.

In [36]:
# ==========================================================
# 5.1 INTERACCIONES — MERCADO
# ==========================================================

df["mkt_ret_x_leverage"] = df["mkt_ret"] * df["leverage"]

df["mkt_ret_x_market_power"] = df["mkt_ret"] * df[MARKET_POWER]

# ==========================================================
# 5.2 INTERACCIONES — CRÉDITO
# ==========================================================

df["credit_level_x_rollover"] = df["credit_level"] * df["rollover_12m"]

df["credit_level_x_market_power"] = df["credit_level"] * df[MARKET_POWER]

# ==========================================================
# 5.3 VALIDACIÓN
# ==========================================================

interaction_vars = [
    "mkt_ret_x_leverage",
    "mkt_ret_x_market_power",
    "credit_level_x_rollover",
    "credit_level_x_market_power"
]

missing_interactions = [v for v in interaction_vars if v not in df.columns]

if missing_interactions:
    raise ValueError(f"Faltan interacciones: {missing_interactions}")

print("✔ Interacciones construidas correctamente")

✔ Interacciones construidas correctamente


## 6) Definición de muestras

En esta sección se definen las muestras utilizadas para la estimación.

Se distinguen tres conjuntos de datos:

1. Muestra principal basada en OAS
2. Muestra alternativa basada en spread construido
3. Muestra alternativa basada en CDS

---

## Principio

Cada variable dependiente define su propia muestra.

No se utiliza una muestra común, ya que esto implicaría perder información innecesariamente.

---

## Objetivo

- maximizar uso de datos
- mantener consistencia dentro de cada especificación
- permitir comparaciones de robustez

In [37]:

# ==========================================================
# 6.1 FUNCIÓN DE MUESTRA
# ==========================================================

def get_sample(df, dep_var, required_vars):
    """
    Devuelve dataframe filtrado para estimación:
    - elimina NA solo en variables relevantes
    """
    cols_needed = [dep_var] + required_vars
    
    df_model = df[cols_needed].copy()
    
    df_model = df_model.dropna()
    
    return df_model

# Variables que SIEMPRE entran
BASE_VARS = (
    FIRM_CONTROLS
    + MACRO_VARS
    + [MARKET_SHOCK]
    + CREDIT_VARS
    + [MARKET_POWER]
)

# agregar interacciones
INTERACTION_VARS = [
    "mkt_ret_x_leverage",
    "mkt_ret_x_market_power",
    "credit_level_x_rollover",
    "credit_level_x_market_power"
]

# ==========================================================
# 6.3 MUESTRA OAS (PRINCIPAL)
# ==========================================================

df_oas = get_sample(
    df,
    DEP_VAR_MAIN,
    BASE_VARS + INTERACTION_VARS
)

print("Muestra OAS:")
print(df_oas.shape)

# ==========================================================
# 6.4 MUESTRA SPREAD
# ==========================================================

df_spread = get_sample(
    df,
    "spread_mean_bps",
    BASE_VARS + INTERACTION_VARS
)

print("\nMuestra Spread:")
print(df_spread.shape)

# ==========================================================
# 6.5 MUESTRA CDS
# ==========================================================

df_cds = get_sample(
    df,
    "cds_5y_mean",
    BASE_VARS + INTERACTION_VARS
)

print("\nMuestra CDS:")
print(df_cds.shape)

# ==========================================================
# 6.6 COMPARACIÓN DE MUESTRAS
# ==========================================================

def sample_stats(df_sample, name):
    n_obs = len(df_sample)
    n_firms = df_sample.index.get_level_values(0).nunique()
    n_time = df_sample.index.get_level_values(1).nunique()
    
    print(f"\n{name}:")
    print(f"Observaciones: {n_obs}")
    print(f"Firmas: {n_firms}")
    print(f"Periodos: {n_time}")

sample_stats(df_oas, "OAS")
sample_stats(df_spread, "Spread")
sample_stats(df_cds, "CDS")

Muestra OAS:
(1652, 19)

Muestra Spread:
(1652, 19)

Muestra CDS:
(839, 19)

OAS:
Observaciones: 1652
Firmas: 26
Periodos: 101

Spread:
Observaciones: 1652
Firmas: 26
Periodos: 101

CDS:
Observaciones: 839
Firmas: 14
Periodos: 100


## 7) Funciones de estimación

En esta sección se definen funciones auxiliares para estimar modelos de panel.

Las funciones permiten:

- estimar modelos con efectos fijos,
- controlar por efectos de tiempo cuando corresponde,
- clusterizar errores por firma,
- estructurar resultados de forma consistente.

---

## Especificación general

Los modelos se estiman mediante PanelOLS con:

- efectos fijos por firma en todos los modelos,
- efectos fijos de tiempo solo en el modelo benchmark (M0),
- errores estándar robustos clusterizados por firma.

In [38]:
# ==========================================================
# 7.1 FUNCIÓN BASE DE ESTIMACIÓN
# ==========================================================

from linearmodels.panel import PanelOLS

def run_panel_model(df, dep_var, exog_vars, add_time_fe=False):
    """
    Estima modelo PanelOLS con:
    - FE firma siempre
    - FE tiempo opcional
    - cluster por firma
    """
    
    y = df[dep_var]
    X = df[exog_vars]
    
    # Modelo
    model = PanelOLS(
        y,
        X,
        entity_effects=True,
        time_effects=add_time_fe
    )
    
    # Fit
    results = model.fit(
        cov_type="clustered",
        cluster_entity=True
    )
    
    return results

# ==========================================================
# 7.2 PREPARACIÓN DE DATASET POR MODELO
# ==========================================================

def prepare_model_data(df, dep_var, exog_vars):
    """
    Selecciona variables relevantes y elimina NA
    SOLO para ese modelo
    """
    
    cols = [dep_var] + exog_vars
    
    df_model = df[cols].dropna()
    
    return df_model

# ==========================================================
# 7.3 FUNCIÓN COMPLETA DE ESTIMACIÓN
# ==========================================================

def estimate_model(df, dep_var, exog_vars, add_time_fe=False):
    
    df_model = prepare_model_data(df, dep_var, exog_vars)
    
    results = run_panel_model(
        df_model,
        dep_var,
        exog_vars,
        add_time_fe=add_time_fe
    )
    
    return results, df_model

# ==========================================================
# 7.4 RESUMEN DE MUESTRA
# ==========================================================

def sample_info(df_model):
    
    n_obs = len(df_model)
    n_firms = df_model.index.get_level_values(0).nunique()
    n_time = df_model.index.get_level_values(1).nunique()
    
    return {
        "n_obs": n_obs,
        "n_firms": n_firms,
        "n_time": n_time
    }

# ==========================================================
# 7.5 TEST RÁPIDO
# ==========================================================

test_vars = [
    "leverage",
    "log_assets",
    "mkt_ret"
]

res_test, df_test = estimate_model(
    df,
    DEP_VAR_MAIN,
    test_vars,
    add_time_fe=False
)

print(res_test.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:               oas_mean   R-squared:                        0.1074
Estimator:                   PanelOLS   R-squared (Between):             -21.434
No. Observations:                2220   R-squared (Within):               0.1074
Date:                Thu, Mar 19 2026   R-squared (Overall):             -16.557
Time:                        19:02:03   Log-likelihood                -1.082e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      87.642
Entities:                          32   P-value                           0.0000
Avg Obs:                       69.375   Distribution:                  F(3,2185)
Min Obs:                       6.0000                                           
Max Obs:                       106.00   F-statistic (robust):             45.238
                            

## 8) Definición de modelos M0–M6

En esta sección se definen las especificaciones empíricas utilizadas en el análisis.

La secuencia de modelos permite identificar progresivamente:

1. un benchmark con efectos fijos completos,
2. un backbone macro-financiero observable,
3. el efecto de shocks de mercado,
4. heterogeneidad en la exposición a shocks de mercado,
5. el efecto de shocks de crédito,
6. heterogeneidad en la exposición a shocks de crédito,
7. un modelo integrado final.

---

## Estructura

- M0: Benchmark (FE firma + FE tiempo)
- M1: Backbone macro-financiero
- M2: Shock de mercado
- M3: Mercado heterogéneo
- M4: Shock de crédito
- M5: Crédito heterogéneo
- M6: Modelo integrado

In [39]:
# ==========================================================
# M0 — BENCHMARK
# ==========================================================

M0_VARS = FIRM_CONTROLS.copy()

M0_CONFIG = {
    "vars": M0_VARS,
    "time_fe": True
}

# ==========================================================
# M1 — BACKBONE MACRO
# ==========================================================

M1_VARS = FIRM_CONTROLS + MACRO_VARS

M1_CONFIG = {
    "vars": M1_VARS,
    "time_fe": False
}

# ==========================================================
# M2 — SHOCK DE MERCADO
# ==========================================================

M2_VARS = M1_VARS + [MARKET_SHOCK]

M2_CONFIG = {
    "vars": M2_VARS,
    "time_fe": False
}

# ==========================================================
# M3 — MERCADO HETEROGÉNEO
# ==========================================================

M3_VARS = M2_VARS + [
    "mkt_ret_x_leverage",
    "mkt_ret_x_market_power"
]

M3_CONFIG = {
    "vars": M3_VARS,
    "time_fe": False
}

# ==========================================================
# M4 — SHOCK DE CRÉDITO
# ==========================================================

M4_VARS = M1_VARS + CREDIT_VARS

M4_CONFIG = {
    "vars": M4_VARS,
    "time_fe": False
}

# ==========================================================
# M5 — CRÉDITO HETEROGÉNEO
# ==========================================================

M5_VARS = M4_VARS + [
    "credit_level_x_rollover",
    "credit_level_x_market_power"
]

M5_CONFIG = {
    "vars": M5_VARS,
    "time_fe": False
}

# ==========================================================
# M6 — MODELO FINAL
# ==========================================================

M6_VARS = (
    FIRM_CONTROLS
    + MACRO_VARS
    + [MARKET_SHOCK]
    + CREDIT_VARS
    + [
        "mkt_ret_x_leverage",
        "credit_level_x_rollover"
    ]
)

M6_CONFIG = {
    "vars": M6_VARS,
    "time_fe": False
}

# ==========================================================
# DICCIONARIO DE MODELOS
# ==========================================================

MODEL_SPECS = {
    "M0": M0_CONFIG,
    "M1": M1_CONFIG,
    "M2": M2_CONFIG,
    "M3": M3_CONFIG,
    "M4": M4_CONFIG,
    "M5": M5_CONFIG,
    "M6": M6_CONFIG,
}


## 9) Estimación de modelos

En esta sección se estiman los modelos M0–M6 utilizando la variable dependiente principal (OAS).

Cada modelo se estima sobre su muestra efectiva, eliminando valores faltantes únicamente en las variables relevantes para cada especificación.

---

## Objetivo

- obtener coeficientes estimados
- comparar especificaciones
- analizar estabilidad de resultados
- preparar tablas para la tesis

In [40]:
# ==========================================================
# 9.1 ESTIMACIÓN DE MODELOS
# ==========================================================

results_dict = {}
sample_dict = {}

for model_name, config in MODEL_SPECS.items():
    
    print(f"\n==============================")
    print(f"Estimando {model_name}")
    print(f"==============================")
    
    vars_model = config["vars"]
    time_fe = config["time_fe"]
    
    res, df_model = estimate_model(
        df,
        DEP_VAR_MAIN,
        vars_model,
        add_time_fe=time_fe
    )
    
    results_dict[model_name] = res
    sample_dict[model_name] = sample_info(df_model)
    
    print(f"Observaciones: {sample_dict[model_name]['n_obs']}")
    print(f"Firmas: {sample_dict[model_name]['n_firms']}")
    print(f"Periodos: {sample_dict[model_name]['n_time']}")

# ==========================================================
# 9.2 RESUMEN INDIVIDUAL
# ==========================================================

for model_name, res in results_dict.items():
    print(f"\n\n########## {model_name} ##########\n")
    print(res.summary) 

# ==========================================================
# 9.3 COMPARACIÓN DE MUESTRAS
# ==========================================================

import pandas as pd

sample_df = pd.DataFrame(sample_dict).T

print(sample_df)


Estimando M0
Observaciones: 1888
Firmas: 26
Periodos: 105

Estimando M1
Observaciones: 1888
Firmas: 26
Periodos: 105

Estimando M2
Observaciones: 1888
Firmas: 26
Periodos: 105

Estimando M3
Observaciones: 1652
Firmas: 26
Periodos: 101

Estimando M4
Observaciones: 1888
Firmas: 26
Periodos: 105

Estimando M5
Observaciones: 1652
Firmas: 26
Periodos: 101

Estimando M6
Observaciones: 1888
Firmas: 26
Periodos: 105


########## M0 ##########

                          PanelOLS Estimation Summary                           
Dep. Variable:               oas_mean   R-squared:                        0.1571
Estimator:                   PanelOLS   R-squared (Between):             -42.504
No. Observations:                1888   R-squared (Within):               0.1419
Date:                Thu, Mar 19 2026   R-squared (Overall):             -32.607
Time:                        19:02:10   Log-likelihood                   -8517.7
Cov. Estimator:             Clustered                                    